# Notebook 03 — Exploratory Analysis

Runs analysis queries against the SQLite database and produces visualizations.

> **Scope & sampling caveat.** The database is built from a **convenience sample** of 990-PF
> filings taken from two IRS release archives (the 2020 and 2023 releases), spanning fiscal
> years ~2018–2022. Because we take the first N filings from each archive rather than a random
> draw, annual totals are dominated by whichever large funders happen to appear (e.g. a single
> big California Endowment filing inflates 2018). So the **pre/post-2020 test below demonstrates
> the methodology — it is not a representative estimate of the national trend.** Candid's
> representative data shows racial-equity funding *rose* after 2020; reproducing that here would
> require random sampling across all chunks of many release years.

In [ ]:
import sys
sys.path.insert(0, '..')

from pathlib import Path
import pandas as pd
import scipy.stats as stats
import matplotlib.pyplot as plt

from src.data_cleaning import get_connection
from src.visualization import (
    plot_funding_over_time,
    plot_top_funders,
    plot_state_choropleth,
    plot_grant_size_distribution,
)

%matplotlib inline
plt.rcParams['figure.dpi'] = 120

DB = Path('../data/racial_equity_grants.sqlite')
conn = get_connection(DB)
PROC = Path('../data/processed'); PROC.mkdir(exist_ok=True)

In [ ]:
grants_df = pd.read_sql('SELECT * FROM grants', conn)
re_grants = grants_df[grants_df['is_racial_equity'] == 1].copy()
print(f"Total grants: {len(grants_df):,} | Racial equity: {len(re_grants):,} "
      f"({len(re_grants)/max(len(grants_df),1):.1%})")

## 1  Summary statistics

In [ ]:
summary = pd.read_sql("""
    SELECT
        COUNT(*) AS total_grants,
        COUNT(DISTINCT funder_ein) AS unique_funders,
        COUNT(DISTINCT recipient_name) AS unique_recipients,
        ROUND(SUM(grant_amount)/1e6, 2) AS total_dollars_m,
        ROUND(AVG(grant_amount)) AS avg_grant_size,
        MIN(tax_year) AS earliest_year,
        MAX(tax_year) AS latest_year
    FROM grants
    WHERE is_racial_equity = 1
""", conn)
summary.T.rename(columns={0: 'value'})

## 2  Time series: racial equity funding by year

In [ ]:
annual = pd.read_sql("""
    SELECT tax_year,
           COUNT(*) AS grant_count,
           ROUND(SUM(grant_amount)/1e6, 2) AS total_dollars_m
    FROM grants
    WHERE is_racial_equity = 1 AND tax_year IS NOT NULL
    GROUP BY tax_year ORDER BY tax_year
""", conn)
print(f"Tax years present in sample: {annual['tax_year'].tolist()}")
annual

In [ ]:
if annual['tax_year'].nunique() > 1:
    fig = plot_funding_over_time(re_grants)
    fig.savefig(PROC / 'fig_funding_over_time.png', bbox_inches='tight')
    plt.show()
else:
    print(f"Only one tax year ({annual['tax_year'].iloc[0]}) in this sample — "
          "time-series chart needs multiple release-year chunks (see Notebook 01).")

## 3  Pre- vs. post-2020 test (runs when the sample spans 2020)

Welch's t-test on annual totals. Requires at least two years on each side of 2020;
otherwise it is reported as not-applicable for this sample.

In [ ]:
pre  = annual[annual['tax_year'] < 2020]['total_dollars_m']
post = annual[annual['tax_year'] >= 2020]['total_dollars_m']

if len(pre) >= 2 and len(post) >= 2:
    t_stat, p_value = stats.ttest_ind(pre, post, equal_var=False)
    print(f"Pre-2020  mean: ${pre.mean():.1f}M/yr (n={len(pre)})")
    print(f"Post-2020 mean: ${post.mean():.1f}M/yr (n={len(post)})")
    print(f"Welch's t = {t_stat:.2f}, p = {p_value:.4f} — "
          f"{'significant' if p_value < 0.05 else 'not significant'} at α=0.05")
else:
    print(f"Not applicable for this sample: need >=2 years on each side of 2020 "
          f"(have pre={len(pre)}, post={len(post)}). "
          "Load more release-year chunks in Notebook 01 to enable this test.")

## 4  Top funders by racial equity giving

In [ ]:
top_funders = pd.read_sql("""
    SELECT g.funder_ein,
           COALESCE(f.name, g.funder_ein) AS funder_name,
           f.state AS funder_state,
           COUNT(*) AS grant_count,
           ROUND(SUM(g.grant_amount)/1e6, 2) AS total_dollars_m
    FROM grants g
    LEFT JOIN foundations f ON g.funder_ein = f.ein
    WHERE g.is_racial_equity = 1
    GROUP BY g.funder_ein
    ORDER BY SUM(g.grant_amount) DESC
    LIMIT 20
""", conn)
top_funders

In [ ]:
# Use funder names where available for a readable chart
re_named = re_grants.merge(
    pd.read_sql('SELECT ein, name FROM foundations', conn),
    left_on='funder_ein', right_on='ein', how='left'
)
re_named['funder_label'] = re_named['name'].fillna(re_named['funder_ein'])
fig = plot_top_funders(re_named, funder_col='funder_label',
                       n=min(20, re_named['funder_label'].nunique()))
fig.savefig(PROC / 'fig_top_funders.png', bbox_inches='tight')
plt.show()

## 5  Geographic distribution (recipient state)

In [ ]:
by_state = pd.read_sql("""
    SELECT recipient_state,
           COUNT(*) AS grant_count,
           ROUND(SUM(grant_amount)/1e6, 2) AS total_dollars_m
    FROM grants
    WHERE is_racial_equity = 1 AND recipient_state IS NOT NULL
    GROUP BY recipient_state
    ORDER BY SUM(grant_amount) DESC
""", conn)
by_state.head(10)

In [ ]:
try:
    fig_map = plot_state_choropleth(re_grants)
    fig_map.write_html(str(PROC / 'fig_state_map.html'))
    fig_map.show()
except Exception as e:
    print('Choropleth skipped:', e)

## 6  Grant size distribution

In [ ]:
fig = plot_grant_size_distribution(re_grants)
fig.savefig(PROC / 'fig_grant_size_dist.png', bbox_inches='tight')
plt.show()

## 7  Issue-area breakdown (grant-purpose keywords)

Since 990-PF grants lack recipient EINs (so no NTEE codes), we classify issue areas from
the free-text grant purpose — the same field used for racial-equity tagging.

In [ ]:
issue = pd.read_sql("""
    SELECT CASE
        WHEN LOWER(grant_purpose) LIKE '%voting%' OR LOWER(grant_purpose) LIKE '%civil right%' THEN 'Voting / Civil Rights'
        WHEN LOWER(grant_purpose) LIKE '%immigr%'      THEN 'Immigration'
        WHEN LOWER(grant_purpose) LIKE '%health%'      THEN 'Health'
        WHEN LOWER(grant_purpose) LIKE '%educat%'      THEN 'Education'
        WHEN LOWER(grant_purpose) LIKE '%housing%'     THEN 'Housing'
        WHEN LOWER(grant_purpose) LIKE '%justice%'     THEN 'Justice Reform'
        WHEN LOWER(grant_purpose) LIKE '%econ%'        THEN 'Economic Development'
        ELSE 'Other / General'
    END AS issue_area,
    COUNT(*) AS grant_count,
    ROUND(SUM(grant_amount)/1e6, 2) AS total_dollars_m
    FROM grants
    WHERE is_racial_equity = 1
    GROUP BY issue_area
    ORDER BY SUM(grant_amount) DESC
""", conn)
issue

## 8  Funder concentration (Herfindahl-Hirschman Index)

HHI > 0.25 indicates high concentration of racial equity dollars among few funders.

In [ ]:
shares = re_grants.groupby('funder_ein')['grant_amount'].sum()
total = shares.sum()
hhi = ((shares / total) ** 2).sum()
print(f"Racial equity funders in sample: {shares.size}")
print(f"HHI (funder concentration): {hhi:.4f}")
print(f"Top {min(5, shares.size)} funders = "
      f"{shares.nlargest(5).sum()/total:.1%} of racial equity dollars")

Proceed to **Notebook 04** for the written findings.